# RQ1 v2.1.1 — Meta-IFD vs Meta-IFD+ — 20 split mới đã khóa trước

Đây là vòng xác nhận **v2.1.1**, khóa sau khi kết quả v1 đã được xem; v2.1 dừng ở lỗi setup trước khi tạo kết quả thực nghiệm.

- Chạy đủ 20 split mới `2001–2020` và 5 model seed `12–16`; không dừng theo p-value.
- Dùng code/cấu hình mặc định chính thức của Meta-IFD. Hai biến thể chia sẻ cùng batch và cùng ICVAE views.
- Khác biệt duy nhất: Meta-IFD+ nối thêm 12 UFE label-free vào mọi view (14 → 26 chiều).
- Kiểm định chính: Nadeau–Bengio trên 20 delta cấp split; Holm được tính sau khi đủ 6 cặp.
- Đây là stability replication trên cùng 2763 node, không phải external/temporal holdout và không thay Bảng A so với paper.
- Notebook hỗ trợ resume: attach output v2.1.1 trước đó; chỉ split khớp toàn bộ protocol/code/data hash mới được dùng lại.

Master protocol SHA-256: `3c50b13557fc897d71787171b2a98be31b8784de4be6fbfa6932ab063ebd1145`.


In [1]:
from pathlib import Path
import csv
import gc
import hashlib
import importlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

import numpy as np
import torch

EXPECTED_SOURCE_SHA256 = {
    "model.py": "86fcd312eb004500afe72eea6a151189224d9d289f930f34819f974ca3970e3e",
    "run.py": "0cf5488678c04809c4f69c4f6779039b3b85cadea3f952da50a44181f59539f8",
    "attention_conv.py": "0080935c16553a59b498c275c1decab96c5d5569a0194f8435cae6ef4db44c1e",
    "icvae_Phish.pkl": "536c13bc765039c7e1d0fa5ec168e9bfbe47ad92aebd156ccc73de6d194d0bf8",
}
SOURCE_DATA_SHA256 = "8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8"
VARIANTS = ["base", "plus"]
VARIANT_LABEL = {"base": "Meta-IFD", "plus": "Meta-IFD+"}
MODEL_SEEDS = [12, 13, 14, 15, 16]
RESPLIT_SEEDS = [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
EXPECTED_COUNTS = {"train": 1657, "validation": 553, "test": 553}
EDGE_STATS_DIM = 12
RELATION_TO_INDEX = {"call": 0, "trans": 1}
BOOTSTRAP_REPETITIONS = 20_000
MINIMUM_PRACTICAL_DELTA = 0.005
ALPHA = 0.05
CONFIG = {'protocol_id': 'group23-rq1-metaifd-matched-20split-replication-v2.1.1', 'status': 'FROZEN AFTER ALL V1 RESULTS; V2.1 SETUP BUG OBSERVED; BEFORE ANY V2.1.1 EXPERIMENTAL RESULT', 'master_protocol_id': 'group23-rq1-all-six-matched-20split-replication-v2.1.1', 'master_protocol_sha256': '3c50b13557fc897d71787171b2a98be31b8784de4be6fbfa6932ab063ebd1145', 'source_data_sha256': '8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8', 'model_pair': {'baseline': 'Meta-IFD', 'plus': 'Meta-IFD+'}, 'dataset': 'HEIG-Phish', 'target_node': 'EOA', 'hidden': 128, 'learning_rate': 0.001, 'contrastive_lambda': 0.1, 'views': 3, 'batch_size': 512, 'fanout': [100, 100], 'weight_decay': 0.0005, 'max_epochs': 1000, 'patience': 50, 'threshold': 0.5, 'shared_configuration': {'hidden': 128, 'learning_rate': 0.001, 'contrastive_lambda': 0.1, 'views': 3, 'batch_size': 512, 'fanout': [100, 100], 'weight_decay': 0.0005, 'max_epochs': 1000, 'patience': 50, 'threshold': 0.5}, 'flat_config_aliases': 'required by unchanged audited Meta-IFD training code; values are asserted equal to shared_configuration during artifact verification', 'configuration_origin': 'official Meta-IFD default for both variants; no tuning in v2.1.1', 'implementation': {'checkpoint_rule': 'official run.py: validation Macro-F1 >= best resets patience; stop when stale > patience; restore best', 'source_hashes': {'model.py': '86fcd312eb004500afe72eea6a151189224d9d289f930f34819f974ca3970e3e', 'run.py': '0cf5488678c04809c4f69c4f6779039b3b85cadea3f952da50a44181f59539f8', 'attention_conv.py': '0080935c16553a59b498c275c1decab96c5d5569a0194f8435cae6ef4db44c1e', 'icvae_Phish.pkl': '536c13bc765039c7e1d0fa5ec168e9bfbe47ad92aebd156ccc73de6d194d0bf8'}, 'matched_views': 'same materialized batches and same ICVAE-generated views for both variants and all five seeds within each split'}, 'matching_rule': 'Within each pair and split, baseline and plus share the fixed configuration, train/validation/test node IDs, model seed, threshold, and materialized sampled batches/views. Plus differs only by receiving the 12-dimensional label-free UFE input/branch.', 'only_difference_between_variants': 'Meta-IFD+ concatenates the same 12 label-free log1p UFE statistics to every row of every view (14 to 26 dimensions)', 'split_seeds': [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 'model_seeds': [12, 13, 14, 15, 16], 'ufe': {'dimensions': 12, 'definition': 'rows/count/sum x in/out x call/trans inside each sampled batch', 'transform': 'log1p', 'uses_labels': False, 'fanout': [100, 100], 'batch_size': 512}, 'selection_and_blinding': {'hyperparameter_tuning_in_v2_1_1': False, 'configurations': 'frozen from v1 validation decisions / official Meta-IFD default', 'test_materialization': 'after validation decisions/checkpoints are frozen within each split', 'threshold': 0.5, 'early_termination_based_on_results': False}, 'inference': {'test': 'two-sided Nadeau-Bengio corrected resampled t-test on 20 split-level paired deltas', 'k': 20, 'df': 19, 'test_train_ratio': '553/1657', 'alpha': 0.05, 'multiplicity': 'Holm correction across the six primary model-pair p-values', 'minimum_practical_delta': 0.005, 'success_rule': 'mean delta >= 0.005, Nadeau-Bengio 95% CI lower bound > 0, and Holm-adjusted p < 0.05'}, 'reporting_commitment': {'report_all_six_pairs': True, 'report_all_twenty_splits': True, 'report_v1_and_v2_1_1_separately': True, 'do_not_pool_v1_and_v2_1_1_post_hoc': True, 'paper_comparison': 'Use the already completed author/paper split as Table A. V2.1.1 is Table B replication and is not a direct numerical comparison with the paper table.', 'limitations': ['random re-splits reuse the same 2763 labeled nodes', 'not an external, temporal, or unseen-dataset holdout', 'PyG neighborhood sampling may differ across sessions, but variants within each pair reuse identical materialized inputs']}, 'code_sha256': '__CODE_SHA256__', 'change_control': 'Any modification after any V2.1.1 experimental result is viewed requires a new version; v1, unexecuted v2, setup-failed v2.1, v2.1.1, and later versions remain distinguishable.'}
CODE_SHA256 = "a0901dba85138cfacf0d6fef20fdcb23152bc32b32bf219bb03286e7c419dcb7"  # SHA-256 of all code cells with this constant replaced by __CODE_SHA256__; written by the generator
CONFIG["code_sha256"] = CODE_SHA256
PROTOCOL_JSON = json.dumps(CONFIG, sort_keys=True, indent=2) + "\n"
PROTOCOL_SHA256 = hashlib.sha256(PROTOCOL_JSON.encode("utf-8")).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_int64(values):
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


try:
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse
except ModuleNotFoundError:
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "pyg_lib", "torch_scatter", "torch_sparse", "-f", wheel_url])
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse

from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle"
device = torch.device("cuda:0")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("/kaggle/working") / f"rq1_v2_1_1_20split_metaifd_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "protocol.json").write_text(PROTOCOL_JSON, encoding="utf-8")
assert sha256_file(run_dir / "protocol.json") == PROTOCOL_SHA256
print("Protocol SHA256:", PROTOCOL_SHA256)

# Đúng snapshot mã công bố đã audit.
source_candidates = [
    p.parent for p in Path("/kaggle/input").rglob("model.py")
    if (p.parent / "run.py").is_file() and (p.parent / "attention_conv.py").is_file()
    and (p.parent / "pretrain_model/icvae_Phish.pkl").is_file()
]
source_candidates = [
    p for p in source_candidates
    if sha256_file(p / "model.py") == EXPECTED_SOURCE_SHA256["model.py"]
    and sha256_file(p / "run.py") == EXPECTED_SOURCE_SHA256["run.py"]
    and sha256_file(p / "attention_conv.py") == EXPECTED_SOURCE_SHA256["attention_conv.py"]  # excludes patched copies from resumed outputs
    and sha256_file(p / "pretrain_model/icvae_Phish.pkl") == EXPECTED_SOURCE_SHA256["icvae_Phish.pkl"]
]
assert len(source_candidates) >= 1, "Không tìm thấy snapshot Meta-IFD-main đã audit"
source_dir = source_candidates[0]
for name, path in {"model.py": source_dir / "model.py", "run.py": source_dir / "run.py",
                   "attention_conv.py": source_dir / "attention_conv.py",
                   "icvae_Phish.pkl": source_dir / "pretrain_model/icvae_Phish.pkl"}.items():
    assert sha256_file(path) == EXPECTED_SOURCE_SHA256[name], name
repo = run_dir / "Meta-IFD-official-copy"
shutil.copytree(source_dir, repo, ignore=shutil.ignore_patterns("data", "__pycache__"))

# Compatibility patch (giống notebook Meta-IFD đã audit): PairTensor hint cho PyG hiện đại.
attention_file = repo / "attention_conv.py"
content = attention_file.read_text(encoding="utf-8")
old_import = "from torch_geometric.typing import EdgeType, Metadata, NodeType, SparseTensor"
new_import = "from torch_geometric.typing import EdgeType, Metadata, NodeType, PairTensor, SparseTensor"
old_propagate = "# propagate_type: (k: Tensor, q: Tensor, v: Tensor, rel: Tensor)"
new_propagate = "# propagate_type: (v: PairTensor)"
assert old_import in content and old_propagate in content
attention_file.write_text(content.replace(old_import, new_import).replace(old_propagate, new_propagate), encoding="utf-8")

os.chdir(repo)
sys.path.insert(0, str(repo))
for module_name in ("model", "attention_conv", "icvae", "rcvae"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
official_model_module = importlib.import_module("model")
HMSL = official_model_module.HMSL
assert Path(official_model_module.__file__).resolve() == (repo / "model.py").resolve()

data_candidates = [p for p in Path("/kaggle/input").rglob("data.pt")
                   if "phish/processed/data.pt" in str(p).replace("\\", "/").lower()]
assert len(data_candidates) == 1, [str(p) for p in data_candidates]
data_sha256 = sha256_file(data_candidates[0])
assert data_sha256 == SOURCE_DATA_SHA256
loaded = torch.load(data_candidates[0], map_location="cpu", weights_only=False)
data = loaded[0] if isinstance(loaded, tuple) and len(loaded) == 2 else loaded
assert isinstance(data, HeteroData)
del loaded
gc.collect()
assert set(data.node_types) == {"EOA", "CA"}
assert data["EOA"].x.shape[1] == data["CA"].x.shape[1] == 14
labeled_mask = (data["EOA"].y == 0) | (data["EOA"].y == 1)
labeled_ids = torch.where(labeled_mask)[0].numpy()
labeled_y = data["EOA"].y[labeled_mask].numpy()
assert len(labeled_ids) == 2763 and int(labeled_y.sum()) == 1206
fanout_by_relation = {edge_type: CONFIG["fanout"] for edge_type in data.edge_types}
all_edge_types = list(data.edge_types)
edge_type_to_id = {edge_type: idx for idx, edge_type in enumerate(all_edge_types)}
assert len(all_edge_types) == 6

(repo / "rcvae.py").write_text("from icvae import ICVAE, Encoder, Decoder\nRCVAE = ICVAE\n", encoding="utf-8")
sys.modules.pop("rcvae", None)
importlib.invalidate_caches()
import rcvae
icvae_model = torch.load(repo / "pretrain_model/icvae_Phish.pkl", map_location="cpu", weights_only=False).to(device)
icvae_model.eval()

# Resume: nạp kết quả split đã xong từ output phiên trước (nếu được attach làm input) với cùng protocol hash.
previous_split_results = {}
for path in Path("/kaggle/input").rglob("split_result.json"):
    try:
        item = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if item.get("protocol_sha256") == PROTOCOL_SHA256 and item.get("code_sha256") == CODE_SHA256:
        previous_split_results[item["split_key"]] = (item, path.parent)  # full integrity check in reuse_or_run
print("Resumable splits found:", sorted(previous_split_results))
print("Torch/PyG:", torch.__version__, torch_geometric.__version__, "| GPU:", torch.cuda.get_device_name(0))
print("Output:", run_dir)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 44.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 104.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 47.0 MB/s eta 0:00:00
Protocol SHA256: 6c4a8f4b70b8b11ee86016b5116b0a2a2a391ba60a705986cb5c9623774f41ca
Resumable splits found: []
Torch/PyG: 2.10.0+cu128 2.8.0.post1 | GPU: Tesla T4
Output: /kaggle/working/rq1_v2_1_1_20split_metaifd_20260926T003807408164Z


## Hàm dùng chung: batch và ICVAE views khớp tuyệt đối trong từng cặp


In [2]:
import torch.nn.functional as F
from scipy.stats import binomtest, t as student_t, wilcoxon
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

EMPTY_CONTRASTIVE_NODE_TYPES = {"CA": 0, "EOA": 0}


def safe_contrast_module(self, ca_hidden_ls, eoa_hidden_ls):
    """Same patch as the audited Meta-IFD notebook: a node type absent from a batch contributes 0, not NaN."""
    def one_type(hidden_ls, node_type):
        if not hidden_ls or hidden_ls[0].shape[1] == 0:
            EMPTY_CONTRASTIVE_NODE_TYPES[node_type] += 1
            return None
        term = self.loss_co(self.attention([z[1] for z in hidden_ls], node_type),
                            self.attention([z[2] for z in hidden_ls], node_type),
                            self.attention([z[0] for z in hidden_ls], node_type))
        if not torch.isfinite(term):
            raise RuntimeError(f"Non-finite {node_type} contrastive term")
        return term
    terms = [v for v in (one_type(ca_hidden_ls, "CA"), one_type(eoa_hidden_ls, "EOA")) if v is not None]
    if not terms:
        return torch.zeros((), device=next(self.parameters()).device)
    return torch.stack(terms).sum()


HMSL.contrast_module = safe_contrast_module


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_ufe_by_node_type(sampled_batch):
    """12 label-free statistics per node inside the sampled batch (identical to the audited Meta-IFD+)."""
    result = {nt: sampled_batch[nt].x.new_zeros((sampled_batch[nt].num_nodes, EDGE_STATS_DIM)) for nt in ("EOA", "CA")}
    for edge_type in sampled_batch.edge_types:
        source_type, relation, destination_type = edge_type
        store = sampled_batch[edge_type]
        if store.edge_index.numel() == 0:
            continue
        assert hasattr(store, "edge_attr") and store.edge_attr.shape[1] >= 2
        attributes = torch.nan_to_num(store.edge_attr[:, :2].to(dtype=sampled_batch[source_type].x.dtype),
                                      nan=0.0, posinf=0.0, neginf=0.0).clamp_min_(0.0)
        src, dst = store.edge_index[0], store.edge_index[1]
        ones = torch.ones(src.numel(), dtype=attributes.dtype, device=attributes.device)
        column = RELATION_TO_INDEX[relation] * 6
        result[destination_type][:, column + 0].index_add_(0, dst, ones)
        result[source_type][:, column + 1].index_add_(0, src, ones)
        result[destination_type][:, column + 2].index_add_(0, dst, attributes[:, 0])
        result[source_type][:, column + 3].index_add_(0, src, attributes[:, 0])
        result[destination_type][:, column + 4].index_add_(0, dst, attributes[:, 1])
        result[source_type][:, column + 5].index_add_(0, src, attributes[:, 1])
    result = {nt: torch.log1p(v) for nt, v in result.items()}
    assert all(bool(torch.isfinite(v).all()) for v in result.values())
    return result


@torch.no_grad()
def make_shared_views(sampled_batch):
    """One ICVAE realization shared by both variants: base views are 14-D; UFE is stored separately."""
    base_views = {"EOA": [], "CA": []}
    for _ in range(CONFIG["views"]):
        generated = {"EOA": [], "CA": []}
        for edge_type in all_edge_types:
            node_type = edge_type[0]
            num_nodes = sampled_batch[node_type].num_nodes
            edge_onehot = torch.zeros((num_nodes, len(all_edge_types)), device=device, dtype=torch.float32)
            edge_onehot[:, edge_type_to_id[edge_type]] = 1.0
            z = torch.randn((num_nodes, icvae_model.latent_size), device=device)
            augmented = icvae_model.inference(z, sampled_batch[node_type].x, edge_onehot)
            denominator = augmented.sum(dim=1, keepdim=True)
            augmented = torch.where(denominator != 0, augmented / denominator, torch.zeros_like(augmented))
            generated[node_type].append(augmented)
        for node_type in ("EOA", "CA"):
            base_views[node_type].append(torch.stack(generated[node_type] + [sampled_batch[node_type].x]).cpu())
    ufe = {nt: v.cpu() for nt, v in compute_ufe_by_node_type(sampled_batch).items()}
    return base_views, ufe


def fingerprint(items):
    digest = hashlib.sha256()
    for batch, base_views, ufe in items:
        for nt in ("EOA", "CA"):
            digest.update(batch[nt].n_id.detach().cpu().numpy().astype(np.int64, copy=False).tobytes())
            for view in base_views[nt]:
                digest.update(view.detach().cpu().numpy().astype(np.float32, copy=False).tobytes())
            digest.update(ufe[nt].detach().cpu().numpy().astype(np.float32, copy=False).tobytes())
        for et in sorted(batch.edge_types):
            digest.update(batch[et].edge_index.detach().cpu().numpy().astype(np.int64, copy=False).tobytes())
    return digest.hexdigest()


def materialize_split(mask, sampler_seed, view_seed, expected, shuffle):
    random.seed(sampler_seed); np.random.seed(sampler_seed)
    torch.manual_seed(sampler_seed); torch.cuda.manual_seed_all(sampler_seed)
    loader = NeighborLoader(data, input_nodes=("EOA", mask), num_neighbors=fanout_by_relation,
                            batch_size=CONFIG["batch_size"], shuffle=shuffle, num_workers=0)
    batches = [batch.cpu() for batch in loader]
    torch.manual_seed(view_seed); torch.cuda.manual_seed_all(view_seed)
    items, ids = [], []
    for batch in batches:
        n = int(batch["EOA"].batch_size)
        assert bool(torch.all((batch["EOA"].y[:n] == 0) | (batch["EOA"].y[:n] == 1)))
        ids.extend(batch["EOA"].n_id[:n].tolist())
        batch["EOA"].y[n:] = -1
        if "CA" in batch.node_types and hasattr(batch["CA"], "y"):
            batch["CA"].y[:] = -1
        gpu_batch = batch.to(device)
        base_views, ufe = make_shared_views(gpu_batch)
        batch = batch.cpu()
        items.append((batch, base_views, ufe))
        del gpu_batch
        torch.cuda.empty_cache()
    assert len(ids) == len(set(ids)) == expected
    assert set(ids) == set(torch.where(mask)[0].tolist())
    return items, ids


def variant_inputs(variant, base_views, ufe):
    views = {}
    for nt in ("EOA", "CA"):
        views[nt] = []
        for view in base_views[nt]:
            view = view.to(device)
            if variant == "plus":
                view = torch.cat([view, ufe[nt].to(device).unsqueeze(0).expand(view.shape[0], -1, -1)], dim=-1)
                assert view.shape[-1] == 26
            else:
                assert view.shape[-1] == 14
            views[nt].append(view)
    return views


def metrics(labels, scores, loss=None, seconds=None):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    predictions = (scores >= CONFIG["threshold"]).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    result = {
        "pr_auc": float(average_precision_score(labels, scores)), "roc_auc": float(roc_auc_score(labels, scores)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "binary_f1": float(f1_score(labels, predictions, zero_division=0)),
        "micro_f1": float(f1_score(labels, predictions, average="micro")),
        "macro_f1": float(f1_score(labels, predictions, average="macro")),
        "mcc": float(matthews_corrcoef(labels, predictions)), "fpr": float(fp / (fp + tn)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    if loss is not None:
        result["loss"] = float(loss)
    if seconds is not None:
        result["seconds"] = float(seconds)
    return result


def run_items(model, variant, items, optimizer=None, collect_rows=False):
    training = optimizer is not None
    model.train(training)
    labels_all, scores_all, losses, rows = [], [], [], []
    if device.type == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    for cpu_batch, base_views, ufe in items:
        batch = cpu_batch.to(device)
        views = variant_inputs(variant, base_views, ufe)
        n = int(batch["EOA"].batch_size)
        labels = batch["EOA"].y[:n]
        try:
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(training):
                logits, contrastive = model(views, batch.edge_index_dict)
                loss = F.cross_entropy(logits[:n], labels) + CONFIG["contrastive_lambda"] * contrastive
                if not torch.isfinite(loss):
                    raise RuntimeError("Non-finite Meta-IFD loss")
                if training:
                    loss.backward()
                    optimizer.step()
            probabilities = torch.softmax(logits[:n].detach(), dim=1)[:, 1]
            label_values = labels.detach().cpu().tolist()
            score_values = probabilities.detach().cpu().tolist()
            labels_all.extend(label_values)
            scores_all.extend(score_values)
            losses.append(float(loss.detach().cpu()) * n)
            if collect_rows:
                ids = batch["EOA"].n_id[:n].detach().cpu().tolist()
                rows.extend({"node_id": int(i), "label": int(l), "score": float(s),
                             "prediction": int(s >= CONFIG["threshold"]), "threshold": CONFIG["threshold"]}
                            for i, l, s in zip(ids, label_values, score_values))
        finally:
            del batch, views
    if device.type == "cuda":
        torch.cuda.synchronize()
    return metrics(labels_all, scores_all, loss=sum(losses) / len(labels_all),
                   seconds=time.perf_counter() - started), rows


def build_model(variant, seed, template_items):
    """Materialise lazy Linear(-1, .) layers, then re-initialise every parameter from `seed`."""
    set_seed(seed)
    model = HMSL(hidden=CONFIG["hidden"], out_channels=2, data=template_items[0][0],
                 concat=CONFIG["views"], target_node=CONFIG["target_node"]).to(device)
    batch = template_items[0][0].to(device)
    with torch.no_grad():
        model.eval()
        model(variant_inputs(variant, template_items[0][1], template_items[0][2]), batch.edge_index_dict)
    del batch
    set_seed(seed)
    model.reset_parameters()
    return model


def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def write_csv(path, rows):
    assert rows
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def train_one(variant, seed, train_items, validation_items, folder):
    model = build_model(variant, seed, train_items)
    parameters = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    best_macro, best_epoch, best_state, stale, history = -1.0, 0, None, 0, []
    started = time.perf_counter()
    for epoch in range(1, CONFIG["max_epochs"] + 1):
        train_result, _ = run_items(model, variant, train_items, optimizer=optimizer)
        val_result, _ = run_items(model, variant, validation_items)
        if val_result["macro_f1"] >= best_macro:  # official Meta-IFD rule
            best_macro, best_epoch, best_state, stale = val_result["macro_f1"], epoch, clone_state(model), 0
        else:
            stale += 1
        history.append({"epoch": epoch, "train_loss": train_result["loss"], "train_macro_f1": train_result["macro_f1"],
                        "validation_loss": val_result["loss"], "validation_macro_f1": val_result["macro_f1"],
                        "validation_pr_auc": val_result["pr_auc"], "validation_fpr": val_result["fpr"],
                        "best_epoch": best_epoch, "stale_epochs": stale})
        if stale > CONFIG["patience"]:
            break
    training_seconds = time.perf_counter() - started
    model.load_state_dict(best_state, strict=True)
    val_final, _ = run_items(model, variant, validation_items)
    assert abs(val_final["macro_f1"] - best_macro) < 5e-3, (val_final["macro_f1"], best_macro)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{variant}_seed_{seed}.pt"
    torch.save({"state_dict": best_state, "variant": variant, "seed": seed, "best_epoch": best_epoch,
                "trained_epochs": len(history), "parameter_count": parameters,
                "protocol_sha256": PROTOCOL_SHA256, "source_data_sha256": data_sha256}, path)
    write_csv(folder / f"history_{variant}_seed_{seed}.csv", history)
    best_row = history[best_epoch - 1]
    row = {"variant": variant, "seed": seed, "best_epoch": best_epoch, "trained_epochs": len(history),
           "training_seconds": training_seconds, "parameter_count": parameters,
           "train_macro_f1_at_best": best_row["train_macro_f1"],
           **{f"validation_{k}": v for k, v in val_final.items()}, "checkpoint_sha256": sha256_file(path)}
    del model, optimizer, best_state
    gc.collect()
    torch.cuda.empty_cache()
    return row, path


def evaluate_checkpoint(variant, seed, path, expected_hash, test_items, template_items):
    assert sha256_file(path) == expected_hash
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    assert checkpoint["variant"] == variant and checkpoint["seed"] == seed
    assert checkpoint["protocol_sha256"] == PROTOCOL_SHA256
    model = build_model(variant, seed, template_items)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    assert sum(p.numel() for p in model.parameters()) == checkpoint["parameter_count"]
    result, rows = run_items(model, variant, test_items, collect_rows=True)
    del model, checkpoint
    gc.collect()
    torch.cuda.empty_cache()
    return result, rows


def mean_macro(score_matrix, labels, sample):
    truth = labels[sample].astype(bool)
    pred = score_matrix[:, sample] >= CONFIG["threshold"]
    tp = (pred & truth).sum(1); fp = (pred & ~truth).sum(1)
    fn = (~pred & truth).sum(1); tn = (~pred & ~truth).sum(1)
    f1_pos = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(len(tp)), where=(2 * tp + fp + fn) > 0)
    f1_neg = np.divide(2 * tn, 2 * tn + fp + fn, out=np.zeros(len(tn)), where=(2 * tn + fp + fn) > 0)
    return float(np.mean((f1_pos + f1_neg) / 2))


def run_matched_split(split_key, masks, sampler_seeds, view_seeds):
    """Train both variants x 5 seeds, freeze, then (only then) sample test and evaluate."""
    folder = run_dir / split_key
    folder.mkdir(parents=True, exist_ok=False)
    train_items, train_ids = materialize_split(masks["train"], sampler_seeds["train"], view_seeds["train"], EXPECTED_COUNTS["train"], True)
    validation_items, validation_ids = materialize_split(masks["validation"], sampler_seeds["validation"], view_seeds["validation"], EXPECTED_COUNTS["validation"], True)
    assert set(train_ids).isdisjoint(validation_ids)
    shared = {"train_items_sha256": fingerprint(train_items), "validation_items_sha256": fingerprint(validation_items)}
    rows, paths = [], {}
    for variant in VARIANTS:
        for seed in MODEL_SEEDS:
            row, path = train_one(variant, seed, train_items, validation_items, folder)
            rows.append(row); paths[(variant, seed)] = path
            print(f"[{split_key}] {VARIANT_LABEL[variant]} seed={seed} val Macro-F1={row['validation_macro_f1']:.4f} "
                  f"epoch={row['best_epoch']}/{row['trained_epochs']}", flush=True)
    write_csv(folder / "validation_runs.csv", rows)
    decision = {"status": "FROZEN VALIDATION DECISION (both variants)", "split_key": split_key,
                "protocol_sha256": PROTOCOL_SHA256, "source_data_sha256": data_sha256,
                "sampler_seeds": sampler_seeds, "view_seeds": view_seeds,
                "train_target_ids_sha256": sha256_int64(sorted(train_ids)),
                "validation_target_ids_sha256": sha256_int64(sorted(validation_ids)), **shared,
                "checkpoint_sha256": {f"{r['variant']}_seed_{r['seed']}": r["checkpoint_sha256"] for r in rows},
                "test_batches_sampled_before_freeze": False, "test_metrics_computed_before_freeze": False}
    (folder / "frozen_validation_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")

    test_items, test_ids = materialize_split(masks["test"], sampler_seeds["test"], view_seeds["test"], EXPECTED_COUNTS["test"], False)
    assert set(test_ids).isdisjoint(train_ids) and set(test_ids).isdisjoint(validation_ids)
    results, predictions, cube = [], [], {}
    for variant in VARIANTS:
        scores = []
        for seed in MODEL_SEEDS:
            result, prow = evaluate_checkpoint(variant, seed, paths[(variant, seed)],
                                               decision["checkpoint_sha256"][f"{variant}_seed_{seed}"], test_items, train_items)
            assert [r["node_id"] for r in prow] == test_ids
            scores.append([r["score"] for r in prow])
            for r in prow:
                r.update({"split_key": split_key, "variant": variant, "model": VARIANT_LABEL[variant], "seed": seed})
            predictions.extend(prow)
            vrow = next(r for r in rows if r["variant"] == variant and r["seed"] == seed)
            results.append({"split_key": split_key, "variant": variant, "model": VARIANT_LABEL[variant], "seed": seed,
                            "best_epoch": vrow["best_epoch"], "validation_macro_f1": vrow["validation_macro_f1"],
                            **{f"test_{k}": v for k, v in result.items()}})
        cube[variant] = np.asarray(scores)
    labels = np.asarray([r["label"] for r in predictions if r["variant"] == "base" and r["seed"] == MODEL_SEEDS[0]])
    write_csv(folder / "test_results.csv", results)
    write_csv(folder / "test_predictions.csv", predictions)
    split_result = {
        "split_key": split_key, "protocol_sha256": PROTOCOL_SHA256,
        "test_target_ids_sha256": sha256_int64(test_ids), "test_items_sha256": fingerprint(test_items),
        "base_mean_macro_f1": float(np.mean([r["test_macro_f1"] for r in results if r["variant"] == "base"])),
        "plus_mean_macro_f1": float(np.mean([r["test_macro_f1"] for r in results if r["variant"] == "plus"])),
        "base_mean_validation_macro_f1": float(np.mean([r["validation_macro_f1"] for r in results if r["variant"] == "base"])),
        "plus_mean_validation_macro_f1": float(np.mean([r["validation_macro_f1"] for r in results if r["variant"] == "plus"])),
        "decision": decision, "code_sha256": CODE_SHA256, "source_sha256": EXPECTED_SOURCE_SHA256,
        "source_data_sha256": data_sha256,
        "test_target_ids_sorted_sha256": sha256_int64(sorted(test_ids)),
    }
    split_result["delta"] = split_result["plus_mean_macro_f1"] - split_result["base_mean_macro_f1"]
    np.savez_compressed(folder / "score_cube.npz", base=cube["base"], plus=cube["plus"], labels=labels,
                        test_ids=np.asarray(test_ids))
    split_result["score_cube_sha256"] = sha256_file(folder / "score_cube.npz")
    (folder / "split_result.json").write_text(json.dumps(split_result, indent=2), encoding="utf-8")
    # All checkpoints are kept (hashes in frozen_validation_decision.json) so they can be re-verified later.
    del train_items, validation_items, test_items
    gc.collect()
    torch.cuda.empty_cache()
    return split_result, cube, labels


def verify_previous_split(split_key, item, source, masks, sampler_seeds, view_seeds):
    """Return the reloaded cube only if every integrity check passes; otherwise None (split is re-run)."""
    problems = []
    decision = item.get("decision", {})
    checks = {
        "protocol_sha256": (item.get("protocol_sha256"), PROTOCOL_SHA256),
        "code_sha256": (item.get("code_sha256"), CODE_SHA256),
        "source_data_sha256": (item.get("source_data_sha256"), data_sha256),
        "source_sha256": (item.get("source_sha256"), EXPECTED_SOURCE_SHA256),
        "sampler_seeds": (decision.get("sampler_seeds"), sampler_seeds),
        "view_seeds": (decision.get("view_seeds"), view_seeds),
        "train_ids": (decision.get("train_target_ids_sha256"), sha256_int64(sorted(torch.where(masks["train"])[0].tolist()))),
        "validation_ids": (decision.get("validation_target_ids_sha256"), sha256_int64(sorted(torch.where(masks["validation"])[0].tolist()))),
        "test_ids": (item.get("test_target_ids_sorted_sha256"), sha256_int64(sorted(torch.where(masks["test"])[0].tolist()))),
    }
    for name, (got, want) in checks.items():
        if got != want:
            problems.append(name)
    cube_path = source / "score_cube.npz"
    if not cube_path.is_file() or sha256_file(cube_path) != item.get("score_cube_sha256"):
        problems.append("score_cube_sha256")
    for key, digest in decision.get("checkpoint_sha256", {}).items():
        ckpt = source / f"{key}.pt"
        if not ckpt.is_file() or sha256_file(ckpt) != digest:
            problems.append(f"checkpoint:{key}")
    if not problems:
        npz = np.load(cube_path)
        nodes = np.arange(len(npz["labels"]))
        for variant, field in (("base", "base_mean_macro_f1"), ("plus", "plus_mean_macro_f1")):
            if abs(mean_macro(npz[variant], npz["labels"], nodes) - item[field]) > 1e-9:
                problems.append(f"recomputed_{variant}_macro_f1")
        if sha256_int64(sorted(npz["test_ids"].tolist())) != item.get("test_target_ids_sorted_sha256"):
            problems.append("cube_test_ids")
    if problems:
        print(f"[{split_key}] previous result REJECTED ({', '.join(problems)}) -> re-running", flush=True)
        return None
    return {"base": npz["base"], "plus": npz["plus"]}, npz["labels"]


def reuse_or_run(split_key, masks, sampler_seeds, view_seeds):
    if split_key in previous_split_results:
        item, source = previous_split_results[split_key]
        verified = verify_previous_split(split_key, item, source, masks, sampler_seeds, view_seeds)
        if verified is not None:
            shutil.copytree(source, run_dir / split_key)
            print(f"[{split_key}] reused from previous session after full integrity check", flush=True)
            return item, verified[0], verified[1]
    return run_matched_split(split_key, masks, sampler_seeds, view_seeds)


print("Functions ready. Protocol:", PROTOCOL_SHA256)


Functions ready. Protocol: 6c4a8f4b70b8b11ee86016b5116b0a2a2a391ba60a705986cb5c9623774f41ca


## Chạy đủ 20 split v2.1.1 và kiểm định Nadeau–Bengio


In [3]:

# ===== V2: 20 split phân tầng mới, cố định trước khi chạy =====
split_results_b = []
for split_seed in RESPLIT_SEEDS:
    positions = np.arange(len(labeled_ids))
    trainval_pos, test_pos = train_test_split(positions, test_size=0.20, random_state=split_seed, stratify=labeled_y)
    train_pos, validation_pos = train_test_split(trainval_pos, test_size=0.25, random_state=split_seed,
                                                 stratify=labeled_y[trainval_pos])
    assert (len(train_pos), len(validation_pos), len(test_pos)) == (1657, 553, 553)
    masks = {}
    for name, pos in (("train", train_pos), ("validation", validation_pos), ("test", test_pos)):
        m = torch.zeros(data["EOA"].num_nodes, dtype=torch.bool)
        m[torch.from_numpy(labeled_ids[pos])] = True
        masks[name] = m
    assert not bool((masks["train"] & masks["validation"]).any() or (masks["train"] & masks["test"]).any()
                    or (masks["validation"] & masks["test"]).any())
    sampler_seeds = {"train": split_seed * 10 + 1, "validation": split_seed * 10 + 2, "test": split_seed * 10 + 3}
    view_seeds = {"train": split_seed * 10 + 4, "validation": split_seed * 10 + 5, "test": split_seed * 10 + 6}
    result, _, _ = reuse_or_run(f"V2_split_{split_seed}", masks, sampler_seeds, view_seeds)
    split_results_b.append(result)
    print(f"[V2_split_{split_seed}] Meta-IFD {result['base_mean_macro_f1']*100:.2f} | Meta-IFD+ {result['plus_mean_macro_f1']*100:.2f} | "
          f"Δ {result['delta']*100:+.2f} pp", flush=True)

deltas = np.asarray([r["delta"] for r in split_results_b])
k = len(deltas)
assert k == len(RESPLIT_SEEDS) == 20
ratio = EXPECTED_COUNTS["test"] / EXPECTED_COUNTS["train"]
variance = float(deltas.var(ddof=1))
se = float(np.sqrt((1 / k + ratio) * variance))
mean_delta = float(deltas.mean())
if se == 0:
    t_stat, p_nb, ci_nb = (float("inf") if mean_delta else 0.0), (0.0 if mean_delta else 1.0), (mean_delta, mean_delta)
else:
    t_stat = mean_delta / se
    p_nb = float(2 * student_t.sf(abs(t_stat), df=k - 1))
    crit = float(student_t.ppf(0.975, df=k - 1))
    ci_nb = (mean_delta - crit * se, mean_delta + crit * se)
try:
    wilcoxon_p = float(wilcoxon(deltas).pvalue)
except ValueError:
    wilcoxon_p = 1.0
raw_pair_rule_met = bool(mean_delta >= MINIMUM_PRACTICAL_DELTA and ci_nb[0] > 0 and p_nb < ALPHA)
table_b = {
    "role": "fixed-budget v2.1.1 20-split replication; same nodes, not an external holdout",
    "split_seeds": RESPLIT_SEEDS,
    "per_split": [{"split_key": r["split_key"], "base": r["base_mean_macro_f1"], "plus": r["plus_mean_macro_f1"],
                   "delta": r["delta"], "base_val": r["base_mean_validation_macro_f1"],
                   "plus_val": r["plus_mean_validation_macro_f1"]} for r in split_results_b],
    "base_mean_over_splits": float(np.mean([r["base_mean_macro_f1"] for r in split_results_b])),
    "plus_mean_over_splits": float(np.mean([r["plus_mean_macro_f1"] for r in split_results_b])),
    "mean_delta": mean_delta, "sd_delta": float(np.sqrt(variance)),
    "nadeau_bengio": {"se": se, "t": t_stat, "df": k - 1, "p_two_sided_raw": p_nb,
                      "ci95": [ci_nb[0], ci_nb[1]], "ratio_test_train": ratio},
    "descriptive_wilcoxon_p": wilcoxon_p,
    "splits_plus_better": int(np.sum(deltas > 0)), "splits_tied": int(np.sum(deltas == 0)),
    "raw_pair_rule_met": raw_pair_rule_met, "global_holm_pending": True,
}
write_csv(run_dir / "v2_20split_per_split.csv", table_b["per_split"])
final_summary = {"status": "completed fixed-budget v2.1.1 matched Meta-IFD pair on all 20 predetermined splits",
                 "master_protocol_sha256": CONFIG["master_protocol_sha256"],
                 "model_key": "metaifd", "model_label": "Meta-IFD",
                 "protocol_sha256": PROTOCOL_SHA256, "code_sha256": CODE_SHA256,
                 "protocol": CONFIG, "source_data_sha256": data_sha256,
                 "source_sha256": EXPECTED_SOURCE_SHA256, "v2_20split": table_b,
                 "empty_contrastive_node_types": EMPTY_CONTRASTIVE_NODE_TYPES,
                 "all_20_splits_completed": len(split_results_b) == 20,
                 "post_test_selection": False, "protocol_changed_after_results": False,
                 "paper_comparison_in_this_notebook": False}
(run_dir / "final_summary.json").write_text(json.dumps(final_summary, indent=2), encoding="utf-8")
shutil.make_archive(str(run_dir), "zip", run_dir)

print("V2 (20 split mới, fixed-budget replication)")
for r in table_b["per_split"]:
    print(f"  {r['split_key']}: {r['base']*100:.2f} → {r['plus']*100:.2f} (Δ {r['delta']*100:+.2f})")
print(f"  Mean Δ {mean_delta*100:+.2f} pp | NB 95% CI [{ci_nb[0]*100:+.2f}, {ci_nb[1]*100:+.2f}] | p={p_nb:.6f} | "
      f"splits plus better {table_b['splits_plus_better']}/20")
print("Raw pair rule met (Holm across six pairs still pending):", raw_pair_rule_met)
print("Output:", run_dir, "| ZIP:", str(run_dir) + ".zip")


[V2_split_2001] Meta-IFD seed=12 val Macro-F1=0.9835 epoch=72/123
[V2_split_2001] Meta-IFD seed=13 val Macro-F1=0.9816 epoch=59/110
[V2_split_2001] Meta-IFD seed=14 val Macro-F1=0.9871 epoch=50/101
[V2_split_2001] Meta-IFD seed=15 val Macro-F1=0.9871 epoch=73/124
[V2_split_2001] Meta-IFD seed=16 val Macro-F1=0.9853 epoch=40/91
[V2_split_2001] Meta-IFD+ seed=12 val Macro-F1=0.9890 epoch=120/171
[V2_split_2001] Meta-IFD+ seed=13 val Macro-F1=0.9816 epoch=28/79
[V2_split_2001] Meta-IFD+ seed=14 val Macro-F1=0.9853 epoch=40/91
[V2_split_2001] Meta-IFD+ seed=15 val Macro-F1=0.9871 epoch=355/406
[V2_split_2001] Meta-IFD+ seed=16 val Macro-F1=0.9853 epoch=84/135
[V2_split_2001] Meta-IFD 97.65 | Meta-IFD+ 98.31 | Δ +0.66 pp
[V2_split_2002] Meta-IFD seed=12 val Macro-F1=0.9816 epoch=65/116
[V2_split_2002] Meta-IFD seed=13 val Macro-F1=0.9761 epoch=83/134
[V2_split_2002] Meta-IFD seed=14 val Macro-F1=0.9816 epoch=41/92
[V2_split_2002] Meta-IFD seed=15 val Macro-F1=0.9816 epoch=70/121
[V2_split_2